# Compass — Agente de viajes con LLMs
### Trabajo Práctico: Aplicaciones con Modelos de Lenguaje — Opción B: Agentes

Este notebook implementa un agente conversacional basado en LLMs que responde consultas de viaje (clima y conversión de moneda), usando el patrón **ReAct** (Reason + Act) y ejecutando el modelo localmente con **Ollama**.

**Stack:** LangChain + LangGraph + Ollama (`qwen2.5:7b`)


## 1. Setup e instalación de dependencias

Antes de correr este notebook, asegurate de tener:
- Ollama instalado y corriendo
- El modelo descargado: `ollama pull qwen2.5:7b`
- Las dependencias de Python instaladas (ver `requirements.txt`)


In [1]:
# Si estás corriendo esto por primera vez, descomentá la siguiente línea:
# !pip install langchain langchain-ollama langgraph requests

import requests
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain.agents import create_agent


## 2. Definición de las herramientas (tools)

Un LLM por sí solo no puede consultar información en tiempo real ni ejecutar acciones. Las tools son funciones Python comunes que el agente puede invocar de forma autónoma cuando lo considera necesario, en base a la descripción (docstring) de cada una.

### 2.1 Tool: `obtener_clima`

Consulta el clima actual de una ciudad, usando la API pública [Open-Meteo](https://open-meteo.com/) (no requiere API key). Internamente hace dos llamadas: primero geocodifica el nombre de la ciudad a latitud/longitud, y luego consulta el clima para esas coordenadas.


In [2]:
WEATHER_CODES = {
    0: "despejado",
    1: "mayormente despejado",
    2: "parcialmente nublado",
    3: "nublado",
    45: "neblina",
    51: "llovizna ligera",
    61: "lluvia ligera",
    63: "lluvia moderada",
    65: "lluvia fuerte",
    71: "nieve ligera",
    80: "chubascos",
    95: "tormenta",
}

def obtener_coordenadas(ciudad: str):
    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {"name": ciudad}
    response = requests.get(url, params=params)
    data = response.json()

    if "results" not in data or len(data["results"]) == 0:
        return None

    primer_resultado = data["results"][0]
    lat = primer_resultado["latitude"]
    lon = primer_resultado["longitude"]
    nombre_completo = f"{primer_resultado['name']}, {primer_resultado['country']}"
    return lat, lon, nombre_completo


def obtener_clima_por_coordenadas(lat: float, lon: float):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {"latitude": lat, "longitude": lon, "current_weather": True}
    response = requests.get(url, params=params)
    return response.json()


def describir_clima(codigo: int) -> str:
    return WEATHER_CODES.get(codigo, "condición desconocida")


@tool
def obtener_clima(ciudad: str) -> str:
    """Obtiene el clima actual de una ciudad. Recibe el nombre de la ciudad como parámetro."""
    resultado = obtener_coordenadas(ciudad)
    if resultado is None:
        return f"No se encontró la ciudad '{ciudad}'."

    lat, lon, nombre_completo = resultado
    data = obtener_clima_por_coordenadas(lat, lon)

    temp = data["current_weather"]["temperature"]
    viento = data["current_weather"]["windspeed"]
    codigo = data["current_weather"]["weathercode"]
    condicion = describir_clima(codigo)

    return f"En {nombre_completo}: {temp}°C, {condicion}, viento de {viento} km/h."


In [3]:
# Prueba manual de la tool (simula cómo la invocaría el agente)
print(obtener_clima.invoke({"ciudad": "Buenos Aires"}))


En Buenos Aires, Argentina: 10.9°C, mayormente despejado, viento de 6.6 km/h.


### 2.2 Tool: `convertir_moneda`

Convierte un monto entre monedas usando tasas de cambio actuales de la API [Frankfurter](https://frankfurter.dev/).

**Limitación conocida:** Frankfurter sigue datos del Banco Central Europeo, por lo que soporta monedas "fuertes" (USD, EUR, GBP, JPY, etc.) pero no incluye monedas latinoamericanas como ARS.


In [4]:
@tool
def convertir_moneda(monto: float, origen: str, destino: str) -> str:
    """Convierte un monto de una moneda a otra usando tasas de cambio actuales.
    Recibe el monto, el código de moneda de origen (ej: USD) y el código de moneda destino (ej: EUR).
    Soporta principalmente monedas fuertes (USD, EUR, GBP, JPY, etc.), no incluye monedas latinoamericanas como ARS."""
    url = "https://api.frankfurter.dev/v1/latest"
    params = {"amount": monto, "base": origen.upper(), "symbols": destino.upper()}
    response = requests.get(url, params=params)
    data = response.json()

    if "rates" not in data or destino.upper() not in data["rates"]:
        return f"No se pudo convertir de {origen} a {destino}. Verificá que ambos códigos de moneda sean válidos (ej: USD, EUR, GBP, JPY)."

    resultado = data["rates"][destino.upper()]
    return f"{monto} {origen.upper()} equivalen a {resultado} {destino.upper()} (tasa del {data['date']})."


In [5]:
# Prueba manual de la tool
print(convertir_moneda.invoke({"monto": 100, "origen": "USD", "destino": "EUR"}))


100.0 USD equivalen a 85.89 EUR (tasa del 2026-08-28).


## 3. Armado del agente

Usamos `create_agent` (LangChain, apoyado en LangGraph), el enfoque actualmente recomendado por la documentación oficial para construir agentes con ciclo ReAct, en reemplazo del `AgentExecutor` clásico.

El modelo (`qwen2.5:7b` vía Ollama) recibe la lista de tools disponibles; en cada consulta decide de forma autónoma si necesita usar alguna, cuál, y con qué parámetros.


In [6]:
llm = ChatOllama(model="qwen2.5:7b", temperature=0.3, repeat_penalty=1.3)

tools = [obtener_clima, convertir_moneda]

agent = create_agent(
    llm,
    tools,
    system_prompt="Respondé siempre en español, de forma clara y concisa."
)


## 4. Ejemplos de uso

A continuación, tres ejemplos que muestran el ciclo completo **pensar → actuar → observar → responder**, incluyendo la decisión autónoma del agente sobre qué herramienta(s) usar en cada caso.


### Ejemplo 1: Consulta que usa una sola tool (clima)

In [7]:
response = agent.invoke(
    {"messages": [("user", "¿Qué clima hace en Madrid?")]}
)
for mensaje in response["messages"]:
    mensaje.pretty_print()


================================ Human Message =================================

¿Qué clima hace en Madrid?
================================== Ai Message ==================================
Tool Calls:
  obtener_clima (a3ec4886-828e-4aac-93ca-71a653cfd521)
 Call ID: a3ec4886-828e-4aac-93ca-71a653cfd521
  Args:
    ciudad: Madrid
================================= Tool Message =================================
Name: obtener_clima

En Madrid, Spain: 15.7°C, despejado, viento de 3.6 km/h.
================================== Ai Message ==================================

Actualmente en Madrid hace 15.7°C y está despejado con unviento que ronda los 3.6 km/h.


### Ejemplo 2: Consulta que usa una sola tool (conversión de moneda)

In [8]:
response = agent.invoke(
    {"messages": [("user", "¿Cuánto son 100 dólares en euros?")]}
)
for mensaje in response["messages"]:
    mensaje.pretty_print()


================================ Human Message =================================

¿Cuánto son 100 dólares en euros?
================================== Ai Message ==================================
Tool Calls:
  convertir_moneda (a311979b-e85a-40f4-b449-6b4ee018d641)
 Call ID: a311979b-e85a-40f4-b449-6b4ee018d641
  Args:
    destino: EUR
    monto: 100
    origen: USD
================================= Tool Message =================================
Name: convertir_moneda

100.0 USD equivalen a 85.89 EUR (tasa del 2026-08-28).
================================== Ai Message ==================================

100 dólares son aproximadamente 85,89 euros según la tasa actual de cambio para el día 28/08/2026.


### Ejemplo 3: Consulta combinada — el agente decide usar ambas tools

Este ejemplo es el más representativo del comportamiento "agente": una sola pregunta que requiere dos herramientas distintas, invocadas de forma autónoma en el mismo ciclo de razonamiento.


In [9]:
response = agent.invoke(
    {"messages": [("user", "¿Qué clima hace en Madrid y cuánto son 100 dólares en euros?")]}
)
for mensaje in response["messages"]:
    mensaje.pretty_print()


================================ Human Message =================================

¿Qué clima hace en Madrid y cuánto son 100 dólares en euros?
================================== Ai Message ==================================

pecting la información más reciente, primero obtendré el clima actual de Madrid.
Tool Calls:
  obtener_clima (048c42f9-4e4b-48b1-8be9-c00f5ab738ab)
 Call ID: 048c42f9-4e4b-48b1-8be9-c00f5ab738ab
  Args:
    ciudad: Madrid
================================= Tool Message =================================
Name: obtener_clima

En Madrid, Spain: 15.7°C, despejado, viento de 3.6 km/h.
================================== Ai Message ==================================

El clima en Madrid es de 15.7°C y está despejado.

Ahora convertiremos $100 a euros (EUR).
Tool Calls:
  convertir_moneda (a2da3108-016e-48a0-972a-d2bcb8810d1b)
 Call ID: a2da3108-016e-48a0-972a-d2bcb8810d1b
  Args:
    monto: 100
    origen: USD
    destino: EUR
================================= Tool Message =

### Ejemplo extra: manejo de errores / límites conocidos

El agente maneja de forma robusta los casos fuera del alcance de una tool (ej. una moneda no soportada), devolviendo una explicación coherente en vez de inventar un resultado.


In [10]:
response = agent.invoke(
    {"messages": [("user", "¿Cuánto son 1000 pesos argentinos en euros?")]}
)
for mensaje in response["messages"]:
    mensaje.pretty_print()


================================ Human Message =================================

¿Cuánto son 1000 pesos argentinos en euros?
================================== Ai Message ==================================
Tool Calls:
  convertir_moneda (cd8eedef-9982-463e-9c8e-35fceb1a08f5)
 Call ID: cd8eedef-9982-463e-9c8e-35fceb1a08f5
  Args:
    monto: 1000
    origen: ARS
    destino: EUR
================================= Tool Message =================================
Name: convertir_moneda

No se pudo convertir de ARS a EUR. Verificá que ambos códigos de moneda sean válidos (ej: USD, EUR, GBP, JPY).
================================== Ai Message ==================================

Lo siento, pero actualmente no puedo realizar conversiones directas desde pesos argentinos (ARS) a euros (EUR). Para poder hacer esta conversión necesitaría soporte para ARS en la función de cambio monedero. Por ahora solo podemos convertir entre las principales monedas fuertes como USD, EUR y GBP.

Si deseás realizar u

## 5. Conclusiones y análisis crítico

**Limitaciones:**
- El agente no mantiene memoria entre invocaciones independientes (cada `.invoke()` es un ciclo nuevo, sin contexto de preguntas anteriores).
- La conversión de moneda depende de Frankfurter, que no soporta monedas latinoamericanas (ej. ARS).
- La desambiguación de ciudades homónimas (ej. "Buenos Aires" existe en varios países) se resuelve tomando el primer resultado de la API de geocoding, que prioriza la ciudad más poblada — puede no ser siempre la esperada por el usuario.
- Al ser un modelo de 7B parámetros corriendo localmente, se observaron en algunas corridas repeticiones o mezcla de idiomas en la generación; se mitigó ajustando `temperature` y `repeat_penalty`.

**Posibles mejoras:**
- Agregar memoria de conversación entre turnos con un `checkpointer` de LangGraph.
- Sumar una tool de desambiguación de ciudades, pidiendo confirmación del país.
